# DiCo-NLI runner (Kaggle): LLM lớn làm chuyên gia thứ 3 (ý B)

Luật go/no-go: PLAN.md mục 2d, "Thí nghiệm B". Script: `scripts/llm_expert.py` (zero-shot, không train), stack: `scripts/stack_llm3.py`.

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. *Save Version → Save & Run All*.

- GPU 0: EN + EU, GPU 1: ES (mỗi GPU một process, ghi thư mục riêng, gộp ở ô sau). Ước tính 30–60 phút.
- Smoke trước: 20 dòng, vừa tải model 7B một lần.

Kết quả: tải `llm_expert.zip` ở tab *Output*, bỏ vào `kaggle/` (gồm `oof/llm7b_expert/` và `llm7b_expert/seed1/`).

In [ ]:
REPO_URL = 'https://github.com/danh-tc/DiCo-NLI.git'
!git clone --depth 1 $REPO_URL /kaggle/working/dico-nli
%cd /kaggle/working/dico-nli
!git log -1 --oneline
!pip install -q -e .
# --no-deps: không để bitsandbytes kéo torch khác về
!pip install -q --no-deps bitsandbytes
!bash scripts/setup_data.sh
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, bitsandbytes as bnb
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'bnb', bnb.__version__)
assert torch.cuda.is_available(), 'Không có CUDA: bật GPU T4 x2 trong Settings'

In [ ]:
%%bash
# Smoke: 20 dòng mỗi split (cũng là bước tải model, để hai process không tải song song)
set -e
CUDA_VISIBLE_DEVICES=0 DICO_RUNS_DIR=/kaggle/working/smoke python scripts/llm_expert.py --load_4bit --tracks 1 --limit 20 > /kaggle/working/smoke.log 2>&1 || { tail -40 /kaggle/working/smoke.log; exit 1; }
grep -E "unique|scored .* pairs" /kaggle/working/smoke.log
rm -rf /kaggle/working/smoke
echo smoke OK

In [ ]:
%%bash
mkdir -p /kaggle/working/runs/logs
CUDA_VISIBLE_DEVICES=0 python scripts/llm_expert.py --load_4bit --tracks 1 3 --name llm7b_part0 > /kaggle/working/runs/logs/llm_gpu0.log 2>&1 &
CUDA_VISIBLE_DEVICES=1 python scripts/llm_expert.py --load_4bit --tracks 2 --name llm7b_part1 > /kaggle/working/runs/logs/llm_gpu1.log 2>&1 &
wait
for f in /kaggle/working/runs/logs/llm_gpu*.log; do echo "== $f"; grep -v -i warn $f | tail -12; done

In [ ]:
# Gộp hai phần thành một expert `llm7b_expert` (train OOF-layout + dev seed1)
import numpy as np, pandas as pd, shutil, json
from pathlib import Path
R = Path('/kaggle/working/runs')
parts = ['llm7b_part0', 'llm7b_part1']
oof = R / 'oof' / 'llm7b_expert'; oof.mkdir(parents=True, exist_ok=True)
for f in ('train_oof_probs.npy', 'train_oof_swap_probs.npy'):
    np.save(oof / f, np.concatenate([np.load(R / 'oof' / p / f) for p in parts]))
pd.concat([pd.read_csv(R / 'oof' / p / 'train_oof_rows.csv') for p in parts]).to_csv(oof / 'train_oof_rows.csv', index=False)
seed = R / 'llm7b_expert' / 'seed1'; seed.mkdir(parents=True, exist_ok=True)
for p in parts:
    for f in (R / p / 'seed1').iterdir():
        (shutil.copytree if f.is_dir() else shutil.copy)(f, seed / f.name)
    print(p, json.loads((R / p / 'seed1' / 'results.json').read_text())['minutes'], 'min')
print(pd.concat([pd.DataFrame(json.loads((R / p / 'seed1' / 'results.json').read_text())['dev_bias0']) for p in parts]).round(4).to_string(index=False))

In [ ]:
!cd /kaggle/working/runs && zip -qr /kaggle/working/llm_expert.zip oof/llm7b_expert llm7b_expert logs && ls -lh /kaggle/working/llm_expert.zip